### Tenacity - Retrying LLM calls

In [ ]:
import random

def flaky_llm_call(prompt: str):
    if random.random() < 0.5: 
        raise Exception("429: Too many api calls")
    return f"LLM response to: {prompt}"

In [ ]:
# Without tenacity
def call_llm(prompt: str):
    return flaky_llm_call(prompt)

random.seed(1) # on 2 llm call is happening. not on 1 and 4

try:
    print(call_llm("What is gpt"))
except Exception as e:
    print(f"call failed: {str(e)}")

In [ ]:
# With tenacity
from tenacity import retry, stop_after_attempt, wait_fixed

@retry(
    stop= stop_after_attempt(3), # 3 retry calls only
    wait= wait_fixed(2),         # wait for 2 sec after every retry
    reraise= True                # overwrites tenacity's retry error with actual function error 
)
def call_llm_basic_retry(prompt: str):
    print("Attempting call") # one time it is always printed, second time print means error occured
    return flaky_llm_call(prompt)

random.seed(2)

call_llm_basic_retry("Provide me bun")

In [ ]:
# Exceptional backoff with tenacity: wait time increases exponentially
from tenacity import wait_exponential, RetryCallState

def log_retry(retry_state: RetryCallState) -> None:
    wait_time = retry_state.next_action.sleep # tenacity tells us how long it's about to sleep
    print(f"Attempt: {retry_state.attempt_number} failed -- retrying in {wait_time:.1f}s..")

attempts_so_far = {'count':0} # forces first 3 calls to fail, so we can watch backoff escalates

def flaky_llm_call_backoff(prompt str) -> None:
    attempts_so_far['count'] += 1
    if attempts_so_far['count'] < 4:
        raise Exception("429: Too many requests")
    return f"LLM response for: {prompt}"

@retry(
    stop= stop_after_attempt(5), # stop after 5 attempts
    # min wait=2sec, max wait=10sec, multiper means first wait 2sec, 2sec, 4sec, 4sec,....10sec
    wait= wait_exponential(multipier=1, min=2, max=10), 
    before_sleep= log_retry, #printing wait time as defined above
    reraise= True
)
def call_llm_backoff(prompt: str):
    return flaky_llm_call_backoff(prompt)

call_llm_backoff("Explain backoff retry")

In [ ]:
# Retry on specific exceptions
from tenacity import retry_if_exception_type

def log_retry_exception(retry_state: RetryCallState) -> None:
    wait_time = retry_state.next_action.sleep
    print(f"Attempt: {retry_state.attempt_number} failed -- retrying in {wait_time:.1f}s")

class InvalidRequestError(Exception):
    pass

@retry(
    stop= stop_after_attempt(5),
    wait= wait_fixed(1),
    retry= retry_if_exception_type((TimeoutError, InvalidRequestError)), # only retry these two
    before_sleep= log_retry_exception,
    reraise= True
)
def call_llm_custom_exception(prompt: str, error_type:type[Exception]= TimeoutError):
    if error_type:
        raise error_type("Simulated failure")
    return f"LLM response: {prompt}"

# A trainsent error retries 3 times then raises error
try:
    call_llm_custom_exception("test",error_type=TimeoutError)
except TimeoutError:
    print('Give up after retries on a TimeOutError')

try:
    call_llm_custom_exception("test",error_type=InvalidRequestError)
except InvalidRequestError:
    print('Failed immediately on InvalidRequestError -- no retries attempted')

In [ ]:
# Real LLM call with tenacity

@retry(
    stop= 5,
    wait_fixed= wait_exponential(multipler=1, min=2, max=10),
    retry= retry_if_exception_type((openai.RateLimitError, openai.APITimeOutError, openai.APIConnectionError)), # only retry these two
    before_sleep= log_retry_exception,
    reraise= True
)
def call_real_llm(prompt: str, client=openai.OpenAI):
    response = client.chat.completions.create(
        model= 'gpt-4o-mini',
        message= [{'role':'user','content':prompt}]
    )
    return response.choices[0].message.content

client= openai.OpenAI(api_key="sdfagsg")
print(call_real_llm('Say hello',client=client))

In [ ]:
2+2

#### Testing web search tool (TavilySearch)

In [ ]:
from components.tools.websearch_tool import internet_search

In [ ]:
search = internet_search(query="what is the capital of France")

In [ ]:
print(search)

In [ ]:
print(search['query'])

In [ ]:
print(search['results'][0]['content'])

In [ ]:
response = []
for i, r in enumerate(search["results"], 1):
    title   = r.get("title", "Unknown")
    url     = r.get("url", "")
    snippet = r.get("content", "").strip()
    # Keep only the first 300 characters to avoid wall-of-text
    if len(snippet) > 300:
        snippet = snippet[:300].rsplit(" ", 1)[0] + "..."

    response.append(f"{i}. **{title}**\n   {url}\n   {snippet}")

In [ ]:
print(response)

In [ ]:
result = internet_search("Find flights between Mumbai to Kolkata on 25 August 2026")

In [ ]:
result

### Testing flight tool (avaitation stack)

In [ ]:
from components.tools.flight_tool import search_flight

In [ ]:
flight_result = search_flight("flights between BOM to MAA")

In [ ]:
print(flight_result)

In [ ]:
print(flight_result[0])

In [ ]:
another_search = search_flight({
    "dep_iata": "DAC",   # Tokyo
    "arr_iata": "NRT",    # Mumbai
    "flight_date": "2026-08-25"
}, limit=10)

for f in another_search:
    print(f"Airline: {f['airline']['name']}, Flight: {f['flight']['iata']}, "
          f"From: {f['departure']['airport']} → To: {f['arrival']['airport']}, "
          f"Status: {f['flight_status']}")


In [ ]:
print(another_search)

In [ ]:
dep_iata= "DAC",   # Tokyo
arr_iata= "NRT",
route_info = "Global live flights"

if dep_iata and arr_iata:
    route_info = f"Live flights from {dep_iata} to {arr_iata}"
elif dep_iata:
    route_info = f"Live flights from {dep_iata}"
elif arr_iata:
    route_info = f"Live flights to {arr_iata}"

print(route_info)

In [ ]:
import requests

params = {
  'access_key': key
}

api_result = requests.get('https://api.aviationstack.com/v1/flights', params)

api_response = api_result.json()

for flight in api_response['data']:
    if (flight['live']['is_ground'] is False):
        print(u'%s flight %s from %s (%s) to %s (%s) is in the air.' % (
            flight['airline']['name'],
            flight['flight']['iata'],
            flight['departure']['airport'],
            flight['departure']['iata'],
            flight['arrival']['airport'],
            flight['arrival']['iata']))

In [ ]:
api_response['data'][0]

In [ ]:
from components.utils import clean_text,airport_country_matches,get_best_airport_for_country,country_name_to_code

In [ ]:
text = clean_text("Where flights runs from Mumbai for sightseeing")
print(text)

In [ ]:
code = country_name_to_code("Which flight goes to South Korea")
print(code)

In [ ]:
from components.utils import AIRPORTS

In [ ]:
airport_country_matches(airport=AIRPORTS, country_code=code)
#print(f"airport: {airport}, country: {country}")

In [ ]:
from components.tools.flight_tool import search_flights

In [ ]:
res = search_flights("Plan a 7 days Japan trip from Mumbai")
print(res)

In [ ]:
from datetime import datetime, date

custom_date= date(2026,8,26)
print(custom_date)

format_date = now.strftime("%Y-%m-%d")
print("formatted_date", format_date)

In [ ]:
from datetime import datetime
from dateutil import parser

def extract_date_from_query(query: str) -> str:
    """
    Extracts a date from user query and returns it in ISO 8601 format (YYYY-MM-DD).
    """
    try:
        # Use dateutil parser to handle flexible date formats
        dt = parser.parse(query, fuzzy=True)
        return dt.date().isoformat()  # Only date part, e.g. "2026-08-31"
    except Exception as e:
        return f"Error extracting date: {e}"

# Example usage
query = "Create a plan on 31 August 2026."
iso_date = extract_date_from_query(query)
print("Extracted ISO date:", iso_date)


In [ ]:
from datetime import timedelta
query_1 = "Plan a trip to mumbai on 30 august 2026"
query_2 = "Plan a trip to mumbai"

def format_date(query):
    try:
        dt = parser.parse(query, fuzzy=True)
        formatted_date = dt.date().isoformat()
    except:
        default_date = date.today() + timedelta(days=2)
        formatted_date = default_date.isoformat()
    return formatted_date

query_1_date = format_date(query_1)
query_2_date = format_date(query_2)


In [ ]:
print(f"first query: {query_1_date}")
print(f"second query: {query_2_date}")

In [ ]:
from components.graph.state import initial_state

In [ ]:
re = initial_state("new trip")
re

In [ ]:
print(re)

In [ ]:
from components.prompts.final_response_prompt import FINAL_RESPONSE_PROMPT

In [ ]:
FINAL_RESPONSE_PROMPT

In [ ]:
from components.prompts.itineary_prompt import ITINERARY_AGENT_PROMPT

In [ ]:
print(ITINERARY_AGENT_PROMPT)

In [ ]:
from components.graph.workflow import trip_graph

In [ ]:
configs = {'configurable': {'thread_id':'prav'}}

In [ ]:
trip_graph.update_state(
    configs,
    {
        'flight_result':"Scheduled to BOM",
        'hotel_result': "Few available",
        "itineary_result": "not booked"
    }
)

In [ ]:
print("-"*80)
print(f"Initial state:")
print("-"*80)
print("messages: ", re['messages'])
print("user_query: ", re['user_query'])
print("flight_result: ", re['flight_result'])
print("weather_result: ", re['weather_result'])
print("hotel_result: ", re['hotel_result'])
print("itineary_result: ", re['itineary_result'])
print("approved: ", re['approved'])
print("travellers: ", re['travellers'])
print("date: ", re['date'])
print("="*80)
# Fetch the current state snapshot
current_state = trip_graph.get_state(configs)

print("=== Current State Values ===")
# .values contains the actual data (your state keys and their current data)
for key, value in current_state.values.items():
    print(f"{key}: {value}")

# Optional: Print metadata like next steps or pending tasks
print("\n=== State Metadata ===")
print(f"Next Node(s) to execute: {current_state.next}")
print(f"Task ID / Checkpoint ID: {current_state.config['configurable'].get('checkpoint_id')}")

In [1]:
from components.prompts.itineary_prompt import itineary_agent_prompt
from components.prompts.final_response_prompt import response_prompt
from components.graph.state import AgentState

In [5]:
def itineary_agent(state):
    itineary_prompt = itineary_agent_prompt.format(
        user_query = state.get("user_query", ""),
        flight_result = state.get("flight_result", ""),
        weather_result = state.get("weather_result", ""),
        hotel_result = state.get("hotel_result", "")
    )
    return itineary_prompt
itineary_agent_prompt = itineary_agent(AgentState)
print(f"itineary agent prompt: {itineary_agent_prompt}")
print("-"*80)
def final_response(state):
    prompt = response_prompt.format(
        user_uery = state.get("user_query", ""),
        flight_result = state.get("flight_result", ""),
        hotel_result = state.get("hotel_result", ""),
        itinerary = state.get("itineary_result", ""),
        weather_result = state.get("weather_result", "")
    )
    return prompt
response = final_response(AgentState)
print(f"response agent prompt: {response}")

TypeError: descriptor 'get' for 'dict' objects doesn't apply to a 'str' object

In [8]:
from components.prompts.itineary_prompt import itineary_agent_prompt
print(itineary_agent_prompt.template)


    Create a complete travel itinerary.

    User Query:
    {user_query}

    Flight Results:
    {flight_result}

    Hotel Results:
    {hotel_result}

    Weather Results:
    {weather_result}

    Make the itinerary practical, budget-aware, and easy to follow.
    


In [ ]:
from components.graph.state import initial_state

In [3]:
print("initial_state: ", initial_state("how are you"))

initial_state:  {'messages': [], 'user_query': 'how are you', 'flight_result': '', 'weather_result': '', 'hotel_result': '', 'itineary_result': '', 'approved': False, 'travellers': 2, 'date': <class 'str'>}


In [ ]:
state = initial_state("I want to travel to Goa on 30 August")

print(state)
print("DATE:", state["date"])
print("DATE TYPE:", type(state["date"]))

{'messages': [], 'user_query': 'I want to travel to Goa on 30 August', 'flight_result': '', 'weather_result': '', 'hotel_result': '', 'itineary_result': '', 'approved': False, 'travellers': 2, 'date': <class 'str'>}
DATE: <class 'str'>
DATE TYPE: <class 'type'>


In [ ]:
2+2